# Module 1 · Data Orchestration Pipelines

**Track:** AI Engineering Core  
**Objective:** Master the principles and applications of Data Orchestration Pipelines.

---



## 📑 Table of Contents

* [Why This Notebook Is Non-Negotiable](#why-this-notebook-is-non-negotiable)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Why Cron Jobs Fail at Scale](#1-why-cron-jobs-fail-at-scale)
* [2 · Apache Airflow: The Industry Standard](#2-apache-airflow-the-industry-standard)
  * [Core Concepts](#core-concepts)
  * [Airflow Architecture](#airflow-architecture)
  * [Airflow Scheduling Deep Dive](#airflow-scheduling-deep-dive)
  * [Key Airflow Commands](#key-airflow-commands)
* [3 · Dagster: Software-Defined Assets](#3-dagster-software-defined-assets)
  * [Airflow vs Dagster Philosophy](#airflow-vs-dagster-philosophy)
* [4 · Prefect: Pythonic Orchestration](#4-prefect-pythonic-orchestration)
* [5 · Orchestrating Full ML Pipelines](#5-orchestrating-full-ml-pipelines)
  * [The Production ML DAG](#the-production-ml-dag)
* [6 · Choosing the Right Orchestrator](#6-choosing-the-right-orchestrator)
  * [Decision Matrix](#decision-matrix)
  * [vs Kubeflow Pipelines (NB31)](#vs-kubeflow-pipelines-nb31)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Cron vs Orchestrator](#question-1-cron-vs-orchestrator)
  * [Question 2: Airflow vs Dagster](#question-2-airflow-vs-dagster)
  * [Question 3: XCom](#question-3-xcom)
  * [Question 4: Backfill](#question-4-backfill)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Design a DAG](#exercise-1-design-a-dag)
  * [Exercise 2: Implement in Dagster](#exercise-2-implement-in-dagster)
  * [Exercise 3: Failure Handling](#exercise-3-failure-handling)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors write a `train.py` script and run it manually. Seniors build a DAG (Directed Acyclic Graph) that automatically: extracts fresh data → validates it → computes features → trains the model → evaluates it → deploys if the metrics pass — all without human intervention, with retry logic, alerting, and full audit trails.

**Mental Model:** An orchestrator is like an air traffic controller at an airport. Individual planes (tasks) can fly by themselves, but without the controller (orchestrator), there's no coordination of who lands first, no recovery when a runway is blocked, and no way to reroute when things go wrong.

**Common Junior Pitfall:** Using `time.sleep(3600)` in a Python script to wait for upstream data, then running `os.system('python train.py')` sequentially. This has zero retry logic, zero visibility, and one failure silently breaks everything downstream. Use Airflow sensors or Dagster asset dependencies instead.

---

In [ ]:
# Cell 0 — Install Dependencies
# Note: We install lightweight versions. Full Airflow requires Docker (NB02).
!pip install -q apache-airflow dagster dagster-webserver prefect

---
## 1 · Why Cron Jobs Fail at Scale

Every AI team starts with cron. Here's why they abandon it:

| Problem | Cron | Orchestrator |
|---------|------|--------------|
| **Dependencies** | `train.py` runs at 3 AM even if `etl.py` (2 AM) failed | `train` only runs if `etl` succeeded |
| **Retries** | None — failure is silent | Auto-retry with exponential backoff |
| **Monitoring** | Check logs manually (`grep error /var/log/syslog`) | Web UI with DAG status, task duration, logs |
| **Backfills** | Manually re-run for each missed date | One click: "backfill 2024-01-01 to 2024-03-31" |
| **Parameterization** | Hardcoded dates in scripts | Dynamic: `{{ ds }}` for execution date |
| **Alerting** | None built-in | Slack/PagerDuty/email on failure |
| **Concurrency** | Unlimited (can overload DB) | Configurable pools and parallelism limits |

In [ ]:
# Cell 1 — Why cron fails: simulate the dependency problem

import random
import time
from datetime import datetime

def simulate_cron_pipeline():
    """Simulates what happens with cron-based scheduling."""
    print('=== Cron-Based Pipeline (Fragile) ===')
    print(f'2:00 AM — Starting ETL job...')
    
    # Simulate ETL failure (happens ~20% of the time in reality)
    etl_success = random.random() > 0.3
    if not etl_success:
        print(f'2:15 AM — ❌ ETL FAILED: Database connection timeout')
        print(f'          (Error written to /var/log/etl.log — nobody checks until morning)')
    else:
        print(f'2:45 AM — ✅ ETL completed')
    
    # Cron doesn't know ETL failed — it runs training anyway!
    print(f'3:00 AM — Starting Training job... (cron runs this regardless!)')
    if not etl_success:
        print(f'3:01 AM — ❌ TRAINING FAILED: Missing input data')
        print(f'          Training reads STALE data from yesterday!')
        print(f'          Model deployed with yesterday\'s data — silent degradation!')
    else:
        print(f'3:30 AM — ✅ Training completed')
    
    print(f'4:00 AM — Starting Deployment...')
    if not etl_success:
        print(f'4:01 AM — ❌ Stale model deployed! Nobody notices until Monday.')
    else:
        print(f'4:05 AM — ✅ Deployment completed')
    
    return etl_success

random.seed(1)  # Deterministic: this will show the failure case
simulate_cron_pipeline()

print('\n--- With an orchestrator (Airflow/Dagster) ---')
print('2:00 AM — Starting ETL job...')
print('2:15 AM — ❌ ETL FAILED: Database connection timeout')
print('2:15 AM — 🔄 Auto-retry #1 (wait 60s)...')
print('2:16 AM — 🔄 Auto-retry #2 (wait 120s)...')
print('2:18 AM — ✅ ETL succeeded on retry #2')
print('2:18 AM — ✅ Orchestrator triggers Training (dependency satisfied)')
print('2:48 AM — ✅ Training completed → triggers Deployment')
print('2:53 AM — ✅ Fresh model deployed')

---
## 2 · Apache Airflow: The Industry Standard

Airflow is the most widely used orchestrator in production AI/ML. If you learn one tool, learn this.

### Core Concepts

| Concept | Definition | Analogy |
|---------|-----------|--------|
| **DAG** | Directed Acyclic Graph — the pipeline definition | A recipe with ordered steps |
| **Task** | A single unit of work (Python function, SQL, Spark job) | One step in the recipe |
| **Operator** | Template for a type of task | BashOperator, PythonOperator, SparkOperator |
| **Sensor** | Task that waits for an external condition | Wait for a file to appear in S3 |
| **XCom** | Cross-communication — passing data between tasks | Passing a value from step 1 to step 3 |
| **DAG Run** | One execution of a DAG (for a specific date) | Running the recipe for Jan 15th |
| **Backfill** | Re-run DAG for past dates | Re-cook meals for the last 30 days |

### Airflow Architecture

```
┌──────────────────────────────────────────────┐
│                  Airflow                      │
│                                               │
│  [Webserver]  ← Web UI for monitoring DAGs    │
│       ↕                                       │
│  [Scheduler]  ← Decides WHEN to run tasks     │
│       ↕                                       │
│  [Executor]   ← Decides WHERE to run tasks    │
│   (Local / Celery / K8s)                      │
│       ↕                                       │
│  [Metadata DB] ← Stores DAG state (Postgres)  │
│       ↕                                       │
│  [Workers]    ← Actually execute the tasks     │
└──────────────────────────────────────────────┘
```

In [ ]:
# Cell 2 — Airflow DAG: ML Training Pipeline
# This is a COMPLETE, production-ready Airflow DAG.
# In production, save this as dags/ml_training_pipeline.py

airflow_dag_code = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.common.sql.operators.sql import SQLExecuteQueryOperator
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
from airflow.utils.task_group import TaskGroup

# === Default arguments (applied to every task) ===
default_args = {
    "owner": "ml-team",
    "retries": 3,                              # Retry up to 3 times
    "retry_delay": timedelta(minutes=5),        # Wait 5 min between retries
    "retry_exponential_backoff": True,          # 5min, 10min, 20min
    "email": ["ml-alerts@company.com"],
    "email_on_failure": True,                   # Alert on failure
    "execution_timeout": timedelta(hours=2),    # Kill if stuck > 2 hours
}

# === DAG definition ===
with DAG(
    dag_id="ml_training_pipeline",
    default_args=default_args,
    description="Daily ML model retraining pipeline",
    schedule="0 2 * * *",                       # Run daily at 2 AM UTC
    start_date=datetime(2024, 1, 1),
    catchup=False,                              # Don't backfill old dates
    max_active_runs=1,                          # Only 1 run at a time
    tags=["ml", "training", "production"],
) as dag:

    # --- Task 1: Wait for upstream data ---
    wait_for_data = S3KeySensor(
        task_id="wait_for_upstream_data",
        bucket_name="data-lake",
        bucket_key="gold/features/{{ ds }}/",   # Templated: uses run date
        poke_interval=300,                       # Check every 5 minutes
        timeout=3600,                            # Give up after 1 hour
        mode="reschedule",                       # Free up worker slot while waiting
    )

    # --- Task 2: Extract and validate features ---
    def extract_features(**context):
        """Pull features from the data warehouse."""
        import pandas as pd
        from sqlalchemy import create_engine
        
        execution_date = context["ds"]  # Airflow injects the run date
        engine = create_engine("postgresql://...") 
        df = pd.read_sql(f"""
            SELECT * FROM ml_features 
            WHERE feature_date = \'{execution_date}\'
        """, engine)
        
        # Validate (fail the task if data is bad)
        assert len(df) > 1000, f"Too few rows: {len(df)}"
        assert df.isnull().sum().sum() == 0, "Null values found!"
        
        # Save to S3 for the training step
        output_path = f"s3://ml-artifacts/{execution_date}/features.parquet"
        df.to_parquet(output_path)
        
        # Pass path to downstream tasks via XCom
        return output_path
    
    extract = PythonOperator(
        task_id="extract_features",
        python_callable=extract_features,
    )

    # --- Task 3: Train model ---
    def train_model(**context):
        """Train model and log to MLflow (NB15)."""
        import mlflow
        
        # Get features path from upstream task
        features_path = context["ti"].xcom_pull(task_ids="extract_features")
        
        with mlflow.start_run():
            # ... training logic ...
            mlflow.log_metric("accuracy", 0.94)
            mlflow.log_param("execution_date", context["ds"])
            return "models:/churn-model/staging"
    
    train = PythonOperator(
        task_id="train_model",
        python_callable=train_model,
    )

    # --- Task 4: Evaluate model ---
    def evaluate_model(**context):
        """Gate: only deploy if accuracy > threshold."""
        # In production, compare against current production model
        accuracy = 0.94  # Retrieved from MLflow
        threshold = 0.90
        
        if accuracy < threshold:
            raise ValueError(f"Model accuracy {accuracy} below threshold {threshold}")
        print(f"Model passed evaluation: {accuracy} >= {threshold}")
    
    evaluate = PythonOperator(
        task_id="evaluate_model",
        python_callable=evaluate_model,
    )

    # --- Task 5: Deploy model ---
    def deploy_model(**context):
        """Promote model from staging to production."""
        print("Deploying model to production endpoint...")
        # Call K8s API, update model registry, etc.
    
    deploy = PythonOperator(
        task_id="deploy_model",
        python_callable=deploy_model,
    )

    # === Define dependencies (this IS the DAG) ===
    wait_for_data >> extract >> train >> evaluate >> deploy
'''

print(airflow_dag_code)
print('\n' + '='*60)
print('Save this as: dags/ml_training_pipeline.py')
print('The >> operator defines task dependencies.')
print('Airflow parses this file and displays the DAG in its Web UI.')

### Airflow Scheduling Deep Dive

| Schedule | Cron Expression | Meaning |
|----------|----------------|--------|
| Hourly | `0 * * * *` | Every hour at :00 |
| Daily at 2 AM | `0 2 * * *` | Most common for ML pipelines |
| Weekdays only | `0 9 * * 1-5` | Mon-Fri at 9 AM |
| Every 15 min | `*/15 * * * *` | Near-real-time feature refresh |
| Monthly | `0 0 1 * *` | Model retraining (stable domains) |

### Key Airflow Commands

```bash
# Start Airflow (Docker is recommended — see NB02)
docker compose up -d

# Test a single task without running the full DAG
airflow tasks test ml_training_pipeline extract_features 2024-03-15

# Backfill: re-run for a date range
airflow dags backfill ml_training_pipeline \
    --start-date 2024-01-01 \
    --end-date 2024-03-31

# List DAGs
airflow dags list

# Trigger a manual run
airflow dags trigger ml_training_pipeline
```

In [ ]:
# Cell 3 — Airflow TaskFlow API (modern, cleaner syntax)
# Airflow 2.5+ introduced @task decorators — much more Pythonic.

taskflow_code = '''
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    schedule="0 2 * * *",
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["ml"],
)
def ml_feature_pipeline():
    """Modern TaskFlow API — XCom passing is automatic!"""
    
    @task(retries=3)
    def extract(execution_date: str) -> dict:
        """Extract raw data from source systems."""
        return {
            "row_count": 50000,
            "path": f"s3://data/{execution_date}/raw.parquet",
        }
    
    @task()
    def validate(data: dict) -> dict:
        """Validate data quality."""
        assert data["row_count"] > 1000, "Too few rows!"
        data["validated"] = True
        return data
    
    @task()
    def compute_features(data: dict) -> str:
        """Compute ML features and write to feature store."""
        # Feature engineering logic (see NB11)
        output_path = data["path"].replace("raw", "features")
        return output_path
    
    @task()
    def update_feature_store(features_path: str):
        """Push features to Feast (NB11)."""
        print(f"Updating feature store with {features_path}")
    
    # Dependencies are INFERRED from function calls!
    raw_data = extract(execution_date="{{ ds }}")
    validated_data = validate(raw_data)
    features = compute_features(validated_data)
    update_feature_store(features)

# Instantiate the DAG
ml_feature_pipeline()
'''

print('=== Airflow TaskFlow API (Modern Syntax) ===')
print(taskflow_code)
print('Key difference: @task decorator auto-serializes return values as XCom.')
print('Dependencies are inferred from Python function argument passing.')

---
## 3 · Dagster: Software-Defined Assets

Dagster takes a fundamentally different approach. Instead of defining **tasks** (imperative: "do this, then that"), you define **assets** (declarative: "I want this dataset to exist").

### Airflow vs Dagster Philosophy

| | Airflow | Dagster |
|---|---|---|
| Core abstraction | **Task** (a thing to DO) | **Asset** (a thing that EXISTS) |
| DAG defines | Steps to execute | Datasets to materialize |
| Think of it as | "Run this script" | "Ensure this table is fresh" |
| Testing | Hard (tests need Airflow context) | Easy (assets are just Python functions) |
| Local dev | Need Docker + Airflow setup | `dagster dev` — runs instantly |
| Best for | Complex task orchestration | Data-centric ML pipelines |

In [ ]:
# Cell 4 — Dagster Software-Defined Assets for ML

dagster_code = '''
import dagster as dg
import pandas as pd
from datetime import datetime

# === Software-Defined Assets ===
# Each @asset is a DATASET. Dagster figures out the execution order.

@dg.asset(
    description="Raw user events from production database",
    group_name="bronze",
    metadata={"source": "postgresql", "refresh": "daily"},
)
def raw_events() -> pd.DataFrame:
    """Extract raw events from production DB."""
    # In production: pd.read_sql("SELECT * FROM events", engine)
    return pd.DataFrame({
        "user_id": range(1000),
        "event_type": ["purchase"] * 500 + ["view"] * 500,
        "revenue": [49.99] * 500 + [0.0] * 500,
    })

@dg.asset(
    description="Cleaned, validated events",
    group_name="silver",
)
def cleaned_events(raw_events: pd.DataFrame) -> pd.DataFrame:
    """Clean and validate raw events.
    
    Dagster AUTOMATICALLY knows this depends on raw_events
    because of the function parameter name!
    """
    df = raw_events.dropna().drop_duplicates()
    assert len(df) > 100, "Too few rows after cleaning!"
    return df

@dg.asset(
    description="ML feature table: one row per user",
    group_name="gold",
)
def ml_features(cleaned_events: pd.DataFrame) -> pd.DataFrame:
    """Compute ML features from clean events."""
    features = (
        cleaned_events
        .groupby("user_id")
        .agg(
            total_purchases=("event_type", lambda x: (x == "purchase").sum()),
            total_revenue=("revenue", "sum"),
            event_count=("event_type", "count"),
        )
        .reset_index()
    )
    return features

@dg.asset(
    description="Trained churn prediction model",
    group_name="models",
)
def churn_model(ml_features: pd.DataFrame) -> dict:
    """Train churn model on feature table."""
    # In production: sklearn/XGBoost training + MLflow logging
    return {
        "model_type": "xgboost",
        "accuracy": 0.94,
        "feature_count": len(ml_features.columns),
        "training_rows": len(ml_features),
    }

# === Schedules ===
daily_schedule = dg.ScheduleDefinition(
    name="daily_ml_refresh",
    target="*",                    # Materialize all assets
    cron_schedule="0 2 * * *",     # Daily at 2 AM
)

# === Definitions (entry point) ===
defs = dg.Definitions(
    assets=[raw_events, cleaned_events, ml_features, churn_model],
    schedules=[daily_schedule],
)
'''

print('=== Dagster: Software-Defined Assets ===')
print(dagster_code)
print('\nRun locally: dagster dev -f pipeline.py')
print('This opens a web UI at http://localhost:3000 showing the asset graph.')

In [ ]:
# Cell 5 — Dagster asset checks (data quality built-in)
# Unlike Airflow, Dagster has NATIVE data quality checks.

dagster_checks_code = '''
import dagster as dg

@dg.asset_check(asset=ml_features, description="Feature table must have > 500 rows")
def check_minimum_rows(ml_features: pd.DataFrame) -> dg.AssetCheckResult:
    row_count = len(ml_features)
    return dg.AssetCheckResult(
        passed=row_count > 500,
        metadata={"row_count": row_count},
    )

@dg.asset_check(asset=ml_features, description="No null values in features")
def check_no_nulls(ml_features: pd.DataFrame) -> dg.AssetCheckResult:
    null_count = ml_features.isnull().sum().sum()
    return dg.AssetCheckResult(
        passed=null_count == 0,
        metadata={"null_count": null_count},
    )

@dg.asset_check(asset=ml_features, description="Revenue values must be non-negative")
def check_revenue_positive(ml_features: pd.DataFrame) -> dg.AssetCheckResult:
    negative_count = (ml_features["total_revenue"] < 0).sum()
    return dg.AssetCheckResult(
        passed=negative_count == 0,
        metadata={"negative_revenue_rows": int(negative_count)},
    )
'''

print('=== Dagster Asset Checks (Built-in Data Quality) ===')
print(dagster_checks_code)
print('\nAsset checks run AFTER materialization and show results in the UI.')
print('Failed checks can trigger alerts or block downstream assets.')

---
## 4 · Prefect: Pythonic Orchestration

Prefect targets teams that find Airflow too complex and Dagster too opinionated. It's "just Python" with decorators.

| Feature | Airflow | Dagster | Prefect |
|---------|---------|--------|---------|
| Learning curve | ⭐⭐⭐ | ⭐⭐ | ⭐ |
| Local development | Hard (Docker) | Easy | Easiest |
| Production maturity | Highest | Growing | Growing |
| Community | Massive | Growing | Moderate |
| Deployment | Self-hosted, MWAA | Self-hosted, Cloud | Cloud-first |

In [ ]:
# Cell 6 — Prefect: minimal orchestration

prefect_code = '''
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta
import pandas as pd

@task(retries=3, retry_delay_seconds=60, log_prints=True)
def extract_data(date: str) -> pd.DataFrame:
    """Extract data from source system."""
    print(f"Extracting data for {date}")
    # production: pd.read_sql(...)
    return pd.DataFrame({"user_id": range(1000), "revenue": range(1000)})

@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute features. Results are CACHED for 1 hour."""
    print(f"Computing features for {len(df)} rows")
    return df.assign(revenue_tier=pd.qcut(df["revenue"], 4, labels=["low", "mid", "high", "vip"]))

@task()
def train_model(features: pd.DataFrame) -> dict:
    """Train model."""
    print(f"Training on {len(features)} rows")
    return {"accuracy": 0.93, "model_path": "s3://models/latest"}

@flow(name="ML Training Pipeline", log_prints=True)
def ml_pipeline(date: str = "2024-03-15"):
    """Orchestrate the full ML pipeline."""
    data = extract_data(date)
    features = compute_features(data)
    result = train_model(features)
    print(f"Pipeline complete! Accuracy: {result['accuracy']}")
    return result

# Run locally:
# ml_pipeline(date="2024-03-15")

# Deploy to Prefect Cloud (schedule):
# ml_pipeline.serve(
#     name="daily-ml-pipeline",
#     cron="0 2 * * *",
#     tags=["ml", "production"],
# )
'''

print('=== Prefect: Just Python + Decorators ===')
print(prefect_code)
print('\nPrefect\'s killer feature: run locally with `python pipeline.py`,')
print('then deploy to Prefect Cloud with `.serve()` — same code, no changes.')

---
## 5 · Orchestrating Full ML Pipelines

Now let's tie everything together with a realistic end-to-end ML pipeline that connects concepts from across the curriculum.

### The Production ML DAG

```
┌─────────────────┐     ┌───────────────────┐
│ S3 Sensor        │     │ DB Health Check    │
│ (wait for data)  │     │ (NB03)             │
└───────┬─────────┘     └─────────┬─────────┘
        │                         │
        └────────────┬────────────┘
                     ↓
            ┌────────────────┐
            │ Extract Data    │ ← SQL queries (NB05)
            │ (pd.read_sql)   │
            └───────┬────────┘
                    ↓
            ┌────────────────┐
            │ Validate Data   │ ← Great Expectations (NB07)
            │ (schema + dist) │
            └───────┬────────┘
                    ↓
        ┌───────────┴───────────┐
        ↓                       ↓
┌───────────────┐     ┌─────────────────┐
│ Spark Features │     │ Feast Feature    │ ← Feature Store (NB11)
│ (NB08)         │     │ Store Update     │
└───────┬───────┘     └────────┬────────┘
        └───────────┬──────────┘
                    ↓
            ┌────────────────┐
            │ Train Model    │ ← MLflow (NB15)
            │ (XGBoost/PyT)  │
            └───────┬────────┘
                    ↓
            ┌────────────────┐
            │ Evaluate Model │ ← DeepEval (NB37)
            │ (vs production) │
            └───────┬────────┘
                    ↓
            ┌────────────────┐
            │ Deploy Model   │ ← Container registry (NB33)
            │ (if improved)  │
            └───────┬────────┘
                    ↓
            ┌────────────────┐
            │ Notify Team     │ ← Slack / PagerDuty
            └────────────────┘
```

In [ ]:
# Cell 7 — Simulate a complete orchestrated ML pipeline

import time
from datetime import datetime

class PipelineOrchestrator:
    """Simulates what Airflow/Dagster does under the hood."""
    
    def __init__(self, name: str):
        self.name = name
        self.tasks = []
        self.results = {}
    
    def add_task(self, task_id: str, func, depends_on: list = None, retries: int = 3):
        self.tasks.append({
            'task_id': task_id,
            'func': func,
            'depends_on': depends_on or [],
            'retries': retries,
        })
    
    def run(self):
        print(f'\n{"="*60}')
        print(f'DAG: {self.name}')
        print(f'Started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
        print(f'{"="*60}\n')
        
        for task in self.tasks:
            # Check dependencies
            for dep in task['depends_on']:
                if dep not in self.results or self.results[dep]['status'] != 'success':
                    print(f'  ⏭️  SKIPPED: {task["task_id"]} (dependency {dep} failed)')
                    self.results[task['task_id']] = {'status': 'skipped'}
                    continue
            
            # Execute with retries
            for attempt in range(1, task['retries'] + 1):
                try:
                    start = time.time()
                    result = task['func']()
                    duration = time.time() - start
                    print(f'  ✅ {task["task_id"]} — completed in {duration:.1f}s')
                    self.results[task['task_id']] = {'status': 'success', 'result': result}
                    break
                except Exception as e:
                    if attempt < task['retries']:
                        print(f'  🔄 {task["task_id"]} — attempt {attempt} failed: {e} (retrying...)')
                    else:
                        print(f'  ❌ {task["task_id"]} — FAILED after {task["retries"]} attempts: {e}')
                        self.results[task['task_id']] = {'status': 'failed', 'error': str(e)}
        
        # Summary
        success = sum(1 for r in self.results.values() if r['status'] == 'success')
        failed = sum(1 for r in self.results.values() if r['status'] == 'failed')
        skipped = sum(1 for r in self.results.values() if r['status'] == 'skipped')
        print(f'\n{"="*60}')
        print(f'Results: {success}✅  {failed}❌  {skipped}⏭️')
        print(f'{"="*60}')

# === Build the DAG ===
import random
random.seed(42)

pipeline = PipelineOrchestrator('ml_daily_training')

pipeline.add_task('extract_data', lambda: (
    print(f'    Querying PostgreSQL...'),
    time.sleep(0.2),
    {'rows': 50000}
)[-1])

pipeline.add_task('validate_data', lambda: (
    print(f'    Running Great Expectations checks...'),
    time.sleep(0.1),
    {'checks_passed': 12, 'checks_failed': 0}
)[-1], depends_on=['extract_data'])

def train_with_possible_failure():
    """Simulates transient training failures (GPU OOM, etc)."""
    r = random.random()
    if r < 0.3:  # 30% chance of transient failure
        raise RuntimeError('CUDA out of memory (transient)')
    time.sleep(0.3)
    print(f'    Training XGBoost on 50K rows...')
    return {'accuracy': 0.94, 'model_version': 'v2.3'}

pipeline.add_task('train_model', train_with_possible_failure, 
                  depends_on=['validate_data'], retries=3)

pipeline.add_task('evaluate_model', lambda: (
    print(f'    Comparing with production model (accuracy: 0.92)...'),
    time.sleep(0.1),
    {'improved': True, 'delta': '+0.02'}
)[-1], depends_on=['train_model'])

pipeline.add_task('deploy_model', lambda: (
    print(f'    Pushing to model registry + updating K8s deployment...'),
    time.sleep(0.1),
    {'endpoint': 'https://api.company.com/v2/predict'}
)[-1], depends_on=['evaluate_model'])

pipeline.run()

---
## 6 · Choosing the Right Orchestrator

### Decision Matrix

| Factor | Choose Airflow | Choose Dagster | Choose Prefect |
|--------|:---:|:---:|:---:|
| Team has existing Airflow knowledge | ✅ | | |
| Data-centric ML pipelines | | ✅ | |
| Need it running TODAY | | | ✅ |
| Complex task dependencies | ✅ | | |
| Built-in data quality checks | | ✅ | |
| Strong typing and testing | | ✅ | |
| Enterprise (large org) | ✅ | | |
| Startup (small team) | | ✅ | ✅ |
| Cloud-managed options | MWAA, Cloud Composer | Dagster Cloud | Prefect Cloud |

### vs Kubeflow Pipelines (NB31)

| | Airflow/Dagster/Prefect | Kubeflow Pipelines |
|---|---|---|
| Scope | General-purpose orchestration | ML-specific pipelines |
| Runtime | Docker, VMs, K8s | K8s only |
| Task isolation | Optional containers | Every step = container |
| ML features | None built-in | Experiment tracking, model registry |
| When to use | Data pipelines + ML training | Pure ML experimentation at scale |

**Senior Insight:** Most production ML teams use BOTH. Airflow/Dagster handles the data pipeline (ETL → features → validation). KFP handles the ML pipeline (hyperparameter tuning → training → evaluation). Airflow often *triggers* a KFP run as one of its tasks.

---
## Knowledge Check

### Question 1: Cron vs Orchestrator
Your ETL job failed at 2:15 AM and your training job ran at 3:00 AM on stale data. How would Airflow prevent this?

<details>
<summary>Click to reveal answer</summary>

Airflow uses **task dependencies** (`extract >> train`). The training task will NOT execute until the extract task succeeds. Combined with **retries** (e.g., 3 retries with exponential backoff), the extract task would retry automatically. If all retries fail, Airflow marks the DAG run as failed and sends an alert — the training task is never attempted.
</details>

### Question 2: Airflow vs Dagster
What's the fundamental difference between `@task` in Airflow and `@asset` in Dagster?

<details>
<summary>Click to reveal answer</summary>

An Airflow **@task** represents an *action to perform* (imperative). A Dagster **@asset** represents a *dataset that should exist* (declarative). Dagster infers dependencies from function parameter names matching asset names, while Airflow requires explicit dependency declaration (`>>` or function call chains).
</details>

### Question 3: XCom
In Airflow, how does the training task get the file path produced by the extract task?

<details>
<summary>Click to reveal answer</summary>

Via **XCom** (cross-communication). The extract task returns the path, which Airflow serializes to its metadata database. The training task retrieves it with `context['ti'].xcom_pull(task_ids='extract_features')`. With the TaskFlow API (@task decorator), this is automatic — Python function return values become XCom values, and function parameters pull from upstream XCom.
</details>

### Question 4: Backfill
Your feature pipeline was broken for 2 weeks (March 1-14). How do you recompute features for those dates?

<details>
<summary>Click to reveal answer</summary>

Use Airflow's **backfill** command: `airflow dags backfill my_pipeline --start-date 2024-03-01 --end-date 2024-03-14`. This creates one DAG run per date, executing them (potentially in parallel) with the `{{ ds }}` template variable set to each date. This is impossible with cron — you'd need to manually modify scripts and run them 14 times.
</details>

---
## Practical Practice

### Exercise 1: Design a DAG
You're building a recommendation system. Design the Airflow DAG (tasks and dependencies) for:
1. Extract user interaction data from PostgreSQL
2. Extract product catalog from a REST API
3. Join and validate both datasets
4. Compute user-product feature matrix
5. Train collaborative filtering model
6. A/B test against current production model
7. Deploy if improved

Which tasks can run in parallel? Which need sequential dependencies?

### Exercise 2: Implement in Dagster
Convert the above Airflow DAG into Dagster software-defined assets. Use `@dg.asset` decorators with proper dependency inference.

### Exercise 3: Failure Handling
Your daily pipeline has these SLAs:
- Data extraction must complete by 3 AM
- Model training must complete by 5 AM
- If any step fails, alert the on-call engineer via Slack

Write the Airflow `default_args` and DAG configuration to enforce these constraints.

In [ ]:
# ===== EXERCISE SOLUTIONS (try yourself first!) =====

print('=== Exercise 1: DAG Design ===')
print("""
DAG Structure:

  extract_users  ──┐
                   ├──→ join_validate ──→ compute_features ──→ train_model ──→ ab_test ──→ deploy
  extract_catalog ─┘
  
Parallel tasks:
  - extract_users and extract_catalog can run in PARALLEL (no dependency)
  - Everything else is sequential

Airflow syntax:
  [extract_users, extract_catalog] >> join_validate >> compute_features >> train >> ab_test >> deploy
""")

print('\n=== Exercise 2: Dagster Conversion ===')
print("""
@dg.asset(group_name="bronze")
def user_interactions() -> pd.DataFrame:
    return pd.read_sql("SELECT * FROM interactions", engine)

@dg.asset(group_name="bronze")
def product_catalog() -> pd.DataFrame:
    return requests.get("https://api.company.com/products").json()

@dg.asset(group_name="silver")  
def validated_data(user_interactions, product_catalog) -> pd.DataFrame:
    # Dagster auto-infers both dependencies from parameter names!
    return user_interactions.merge(product_catalog, on="product_id")

@dg.asset(group_name="gold")
def feature_matrix(validated_data) -> pd.DataFrame:
    return compute_user_product_features(validated_data)

@dg.asset(group_name="models")
def rec_model(feature_matrix) -> dict:
    return train_collaborative_filtering(feature_matrix)
""")

print('\n=== Exercise 3: Failure Handling ===')
print("""
from datetime import timedelta
from airflow.providers.slack.notifications.slack import send_slack_notification

default_args = {
    "owner": "ml-team",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "execution_timeout": timedelta(hours=2),
    "on_failure_callback": send_slack_notification(
        text="Task {{ ti.task_id }} failed in {{ ti.dag_id }}!",
        channel="#ml-alerts",
    ),
    "sla": timedelta(hours=3),  # Alert if task takes > 3 hours
}

with DAG(
    ...,
    sla_miss_callback=send_slack_notification(
        text="SLA MISS: Pipeline {{ dag.dag_id }} missed its deadline!",
        channel="#ml-oncall",
    ),
)
""")

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Why not cron | No dependencies, retries, visibility, or backfills |
| Airflow | Industry standard — DAGs, tasks, operators, sensors, XCom |
| TaskFlow API | Modern Airflow syntax with @task decorators |
| Dagster | Asset-based (declarative) — focus on WHAT, not HOW |
| Dagster checks | Built-in data quality validation on assets |
| Prefect | Simplest to start — just Python + decorators |
| Choosing | Airflow for enterprise, Dagster for data-centric, Prefect for speed |

**Connections:** Spark jobs this orchestrates → NB08 | Streaming pipelines → NB10 | Feature stores → NB11 | Great Expectations → NB07 | CI/CD integration → NB31 | Model tracking → NB15 (MLflow)  
**Next →** `10_realtime_streaming.ipynb` — Module 1.4 -- Real-Time Event Streaming & Live Semantic Layers